# Where does ridge beat plain least squares? Anchor-count sweep

Question: with the full anchor set (about 36k to 42k word anchors, d about 768) ridge is only 0.04 to 0.55 nats better than plain least squares. Regularisation should matter when the number of anchors n is small or close to the embedding dimension d (least squares is then ill-conditioned). This notebook measures it directly: for n = 100 ... all, it fits plain least squares and ridge on n random anchors and reports the mean cosine between predicted and true receiver rows on 2000 held-out anchors (5 random subsets per n). CPU only, a few minutes.

How to read it: the last column is ridge minus least squares in held-out cosine. If the gap is large for n near d (768 for Pythia-160m, 896 for Qwen2.5-0.5B) and shrinks as n grows, the regularisation hypothesis holds. Held-out cosine is a proxy for direction quality, not NLL.

In [ ]:
%pip install -q transformers

In [ ]:
import numpy as np, torch, gc, json
from transformers import AutoTokenizer, AutoModelForCausalLM
PAIRS = [("openai-community/gpt2", "EleutherAI/pythia-160m"), ("openai-community/gpt2", "Qwen/Qwen2.5-0.5B")]   # receiver, donor
N_GRID = [100, 200, 400, 600, 768, 900, 1200, 2000, 4000, 8000, "all"]   # number of training anchors
N_TEST = 2000          # held-out anchors, never used for fitting
DRAWS = 5              # random subsets per n
LAMS = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)

def unit(x): return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12)
def emb(name):
    m = AutoModelForCausalLM.from_pretrained(name); E = m.get_input_embeddings().weight.detach().float().cpu().numpy().astype(np.float64); del m; gc.collect()
    return E, AutoTokenizer.from_pretrained(name).get_vocab()
def solve(X, Y, lam):
    A = X.T @ X; A[np.diag_indices_from(A)] += lam * np.trace(A) / A.shape[0]; return np.linalg.solve(A, X.T @ Y)
def fit(Dtr, Rtr, Dte, Rte, rng):
    mu_d, mu_r = Dtr.mean(0), Rtr.mean(0); X, Y = Dtr - mu_d, Rtr - mu_r; s = np.linalg.norm(X, axis=1).mean(); X = X / s; Xt = (Dte - mu_d) / s; Yt = Rte - mu_r
    cos = lambda W: float((unit(Xt @ W) * unit(Yt)).sum(1).mean())
    out = {"LS (lambda->0)": cos(np.linalg.lstsq(X, Y, rcond=None)[0])}
    for lam in LAMS: out[f"ridge fixed {lam:g}"] = cos(solve(X, Y, lam))
    perm = rng.permutation(len(X)); nv = min(max(50, len(X) // 10), len(X) // 5); va, tr = perm[:nv], perm[nv:]     # the notebook's rule: lambda chosen on 10% held-out anchors
    best = max(LAMS, key=lambda l: (unit(X[va] @ solve(X[tr], Y[tr], l)) * unit(Y[va])).sum(1).mean()); out["ridge (lambda searched)"] = cos(solve(X, Y, best)); out["_lam"] = best
    return out
rows = []
for rn, dn in PAIRS:
    R, rv = emb(rn); D, dv = emb(dn)
    keys = sorted(k for k in rv if k in dv and k.startswith("Ġ") and k[1:].isalpha() and len(k) >= 3 and rv[k] < len(R) and dv[k] < len(D))
    ri, di = np.array([rv[k] for k in keys]), np.array([dv[k] for k in keys]); print(f"\n{rn} <- {dn}: {len(keys)} shared word anchors, donor dim {D.shape[1]}")
    rng = np.random.default_rng(0); perm = rng.permutation(len(keys)); te, pool = perm[:N_TEST], perm[N_TEST:]
    print(f"{'n train':>8} | " + " | ".join(f"{k:>24}" for k in ["LS (lambda->0)", "ridge fixed 0.01", "ridge (lambda searched)"]) + " | gap (ridge searched - LS)")
    for n in N_GRID:
        nn = len(pool) if n == "all" else n
        if nn > len(pool): continue
        res = []
        for _ in range(DRAWS):
            idx = rng.choice(pool, nn, replace=False); res.append(fit(D[di[idx]], R[ri[idx]], D[di[te]], R[ri[te]], rng))
        mean = lambda k: np.mean([r[k] for r in res]); sd = lambda k: np.std([r[k] for r in res])
        gap = mean("ridge (lambda searched)") - mean("LS (lambda->0)")
        print(f"{nn:>8} | " + " | ".join(f"{mean(k):>17.4f} ±{sd(k):.3f}" for k in ["LS (lambda->0)", "ridge fixed 0.01", "ridge (lambda searched)"]) + f" | {gap:+.4f}")
        rows.append(dict(pair=f"{rn} <- {dn}", n=nn, d=D.shape[1], **{k: mean(k) for k in res[0] if k != "_lam"}, gap=gap))
json.dump(rows, open("ls_vs_ridge_anchor_sweep.json", "w"), indent=1)
print("\nsaved ls_vs_ridge_anchor_sweep.json")
